In [14]:
# import necessary libraries
import pandas as pd
import plotly.graph_objects as go

## Energy balances Eurostat

In [15]:
# Load the CSV
df_energy_balance = pd.read_csv("/Users/charlottewestenberg/Thesis/00data/from_excel/Energy balances - sankey ready.csv")

# Create list of unique nodes
nodes = list(pd.concat([df_energy_balance["Source"], df_energy_balance["Target"]]).unique())

# Create dictionary mapping node name -> index
node_dict = {name: i for i, name in enumerate(nodes)}

# Map names to numeric IDs
df_energy_balance["source_id"] = df_energy_balance["Source"].map(node_dict)
df_energy_balance["target_id"] = df_energy_balance["Target"].map(node_dict)

# Create Sankey
fig = go.Figure(data=[go.Sankey(
    
    node=dict(
        label=nodes,
        pad=20,
        thickness=20
    ),
    
    link=dict(
        source=df_energy_balance["source_id"],
        target=df_energy_balance["target_id"],
        value=df_energy_balance["Value tonne Carbon"]
    )

)])

fig.update_layout(title_text="Carbon Flow Energy Balances EU")

fig.show()

## Biomass 2022 EU

In [35]:
# Load the CSV
df_biomass_2022 = pd.read_csv("/Users/charlottewestenberg/Thesis/00data/from_excel/biomass_uses_and_flows_2022.csv")

# Create list of unique nodes
nodes = list(pd.concat([df_biomass_2022["Source"], df_biomass_2022["Target"]]).unique())

# Create dictionary mapping node name -> index
node_dict = {name: i for i, name in enumerate(nodes)}

# Map names to numeric IDs
df_biomass_2022["source_id"] = df_biomass_2022["Source"].map(node_dict)
df_biomass_2022["target_id"] = df_biomass_2022["Target"].map(node_dict)

# Create Sankey
fig = go.Figure(data=[go.Sankey(
    
    node=dict(
        label=nodes,
        pad=20,
        thickness=20
    ),
    
    link=dict(
        source=df_biomass_2022["source_id"],
        target=df_biomass_2022["target_id"],
        value=df_biomass_2022["Value C (TONNES)"]
    )

)])

fig.update_layout(title_text="Carbon Flow Biomass Balances EU")

fig.show()




## Fossil Carbon (own calculations)

In [ ]:
# Load the CSV
df_fossil = pd.read_csv("/Users/charlottewestenberg/Thesis/00data/from_excel/Sankey diagram fossil.csv")

# create unique node list
all_nodes = pd.concat([df_fossil["Source"], df_fossil["Target"]]).unique()
node_dict = {node: i for i, node in enumerate(all_nodes)}

# map to numeric IDs
df_fossil["source_id"] = df_fossil["Source"].map(node_dict)
df_fossil["target_id"] = df_fossil["Target"].map(node_dict)

carrier_colors = {
    "Crude oil": "rgba(0, 0, 0, 0.6)",       # black
    "Oil": "rgba(0, 0, 0, 0.6)",
    "Gas": "rgba(255, 165, 0, 0.6)",         # orange
    "Hard coal": "rgba(178, 65, 0, 0.8)",    # ugly brown
    "Brown coal": "rgba(139, 69, 19, 0.8)",  # brown
    "Peat": "rgba(160, 82, 45, 0.8)",        # lighter brown
    "Mixed": "rgba(11, 60, 187, 0.4)"      # blue for aggregated flows
}
#calculate outflows per source and target nodes
source_totals = df_fossil.groupby("Source")["Value_tC"].sum()
node_inflow = df_fossil.groupby("Target")["Value_tC"].sum()

labels_with_values = []
for node in all_nodes:
    inflow = node_inflow.get(node, 0)
    outflow = source_totals.get(node, 0)
    # choose what to display
    value = inflow if inflow > 0 else outflow
    if value > 0:
        labels_with_values.append(f"{node}<br>{value/1e6:.1f} MtC")
    else:
        labels_with_values.append(node)

df_fossil["color"] = df_fossil["Carrier"].map(carrier_colors).fillna("rgba(200,200,200,0.4)")

fig = go.Figure(data=[go.Sankey(
    
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels_with_values,
        color="lightgrey"
    ),

    link=dict(
        source=df_fossil["source_id"],
        target=df_fossil["target_id"],
        value=df_fossil["Value_tC"],
        color=df_fossil["color"]
    )
)])

for carrier, color in carrier_colors.items():
    fig.add_trace(go.Scatter(
        x=[None],
        y=[None],
        mode='markers',
        marker=dict(size=10, color=color),
        legendgroup=carrier,
        showlegend=True,
        name=carrier
    ))

fig.update_layout(
    title_text="EU Fossil Carbon Flows (2023)",
    font_size=12
)
fig.update_layout(
    width=1400,
    height=900
)
fig.update_layout(
    showlegend=True,
    legend=dict(
        title="Energy carrier",
        orientation="h",
        y=-0.1
    )
)
fig.update_layout(
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    plot_bgcolor="white",
    paper_bgcolor="white"
)


fig.show()

In [18]:
# balance check
# some flows should not match, inflows/outflows/sinks have only one in or outflow. carbon accumulates there or leaves the system. 

node_inflow = df_fossil.groupby("Target")["Value_tC"].sum()
node_outflow = df_fossil.groupby("Source")["Value_tC"].sum()

balance_df_fossil = pd.DataFrame({
    "inflow": node_inflow,
    "outflow": node_outflow
}).fillna(0)

balance_df_fossil["difference"] = balance_df_fossil["inflow"] - balance_df_fossil["outflow"]

imbalanced_nodes = balance_df_fossil[abs(balance_df_fossil["difference"]) > 1e-3]
print(imbalanced_nodes)

                             inflow       outflow    difference
CO2 captured           1.400136e+04  0.000000e+00  1.400136e+04
Carbon Emissions       6.209556e+08  0.000000e+00  6.209556e+08
Exports                5.477100e+07  0.000000e+00  5.477100e+07
Fossil inflow          8.407547e+08  8.407547e+08  2.851480e+01
Import                 0.000000e+00  6.788781e+08 -6.788781e+08
Indigenous production  0.000000e+00  1.561214e+08 -1.561214e+08
International bunkers  6.587335e+07  0.000000e+00  6.587335e+07
Products?              9.527839e+07  0.000000e+00  9.527839e+07
Released stocks        0.000000e+00  5.755170e+06 -5.755170e+06
Stock                  3.862326e+06  0.000000e+00  3.862326e+06
Transformation         7.818430e+08  7.818430e+08  9.999990e-03


In [19]:
# system wide check

total_inflows = df_fossil[df_fossil["flow_type"] == "Inflow"]["Value_tC"].sum()
total_outflows = df_fossil[df_fossil["flow_type"] == "Outflow"]["Value_tC"].sum()
total_emissions = df_fossil[df_fossil["flow_type"] == "Emissions"]["Value_tC"].sum()
total_stock = df_fossil[df_fossil["flow_type"] == "Stock"]["Value_tC"].sum()
total_sink = df_fossil[df_fossil["flow_type"] == "Sink"]["Value_tC"].sum()

print(total_inflows)
print(total_outflows + total_emissions + total_stock + total_sink)

840754680.8059999
840754652.2812002
